In [1]:
import pandas as pd
import numpy as np

url = ("https://raw.githubusercontent.com/jxchen/Kaggle"
       "/master/Give%20Me%20Some%20Credit/cs-training.csv")
df_raw = pd.read_csv(url, index_col=0)
df = df_raw.rename(columns={
    "SeriousDlqin2yrs":                    "vo_no",
    "RevolvingUtilizationOfUnsecuredLines": "ty_le_su_dung_tin_dung",
    "age":                                  "tuoi",
    "NumberOfTime30-59DaysPastDueNotWorse": "so_lan_tre_30_59_ngay",
    "DebtRatio":                            "ty_le_no",
    "MonthlyIncome":                        "thu_nhap_thang",
    "NumberOfOpenCreditLinesAndLoans":      "so_tai_khoan_vay",
    "NumberOfTimes90DaysLate":              "so_lan_tre_90_ngay",
    "NumberRealEstateLoansOrLines":         "so_tai_khoan_bat_dong_san",
    "NumberOfTime60-89DaysPastDueNotWorse": "so_lan_tre_60_89_ngay",
    "NumberOfDependents":                   "so_nguoi_phu_thuoc"
})

# ── Demo 1: Boolean indexing ──
khach_rui_ro = df[
    (df["so_lan_tre_90_ngay"] > 0) |          # từng trễ 90 ngày
    (df["ty_le_su_dung_tin_dung"] > 0.9)      # gần chạm hạn mức
]
print(f"Số KH rủi ro cao : {len(khach_rui_ro):,}")
print(f"Tỷ lệ vỡ nợ nhóm này : {khach_rui_ro['vo_no'].mean():.1%}")
print(f"Tỷ lệ vỡ nợ toàn dataset: {df['vo_no'].mean():.1%}")

# ── Demo 2: Verify De Morgan ──
khach_rui_ro_dm = df[
    ~((df["so_lan_tre_90_ngay"] == 0) &
      (df["ty_le_su_dung_tin_dung"] <= 0.9))
]
print(f"\nDe Morgan verify: {len(khach_rui_ro) == len(khach_rui_ro_dm)}")
# Phải in True — 2 cách filter cho cùng kết quả

# ── Demo 3: .loc lấy hàng + cột cụ thể ──
# Lấy 5 KH đầu, chỉ lấy 3 cột quan tâm
print(df.loc[1:5, ["tuoi", "thu_nhap_thang", "vo_no"]])

# ── Demo 4: .query() — dễ đọc hơn ──
nguong_tuoi = 60
ket_qua = df.query("tuoi > @nguong_tuoi and so_lan_tre_90_ngay == 0")
print(f"\nKH trên 60 tuổi, chưa trễ hạn: {len(ket_qua):,}")

# ── Demo 5: Sửa giá trị đúng cách Pandas 2.x ──
df_young = df[df["tuoi"] < 30].copy()        # PHẢI .copy()
df_young.loc[:, "nhom_rui_ro"] = "Trẻ"      # OK — không warning

Số KH rủi ro cao : 23,868
Tỷ lệ vỡ nợ nhóm này : 24.1%
Tỷ lệ vỡ nợ toàn dataset: 6.7%

De Morgan verify: True
   tuoi  thu_nhap_thang  vo_no
1    45          9120.0      1
2    40          2600.0      0
3    38          3042.0      0
4    30          3300.0      0
5    49         63588.0      0

KH trên 60 tuổi, chưa trễ hạn: 43,853


In [3]:
# Gợi ý cấu trúc — bạn tự điền điều kiện:
nhom_tre    = df[(df["tuoi"] >= 20) & (df["tuoi"] <= 40)]
nhom_lon    = df[df["tuoi"] > 60]

print(f"Nhóm trẻ (20-40)     — số KH: {len(nhom_tre):,} | tỷ lệ vỡ nợ: {nhom_tre['vo_no'].mean():.1%}")
print(f"Nhóm lớn tuổi (>60)  — số KH: {len(nhom_lon):,} | tỷ lệ vỡ nợ: {nhom_lon['vo_no'].mean():.1%}")

Nhóm trẻ (20-40)     — số KH: 35,096 | tỷ lệ vỡ nợ: 10.4%
Nhóm lớn tuổi (>60)  — số KH: 45,060 | tỷ lệ vỡ nợ: 3.0%


In [7]:
# Nhóm cần cảnh báo (dùng OR như bạn đề xuất lúc nãy):
# Cách 1 — điều kiện gốc
canh_bao_1 = df[
    (df["so_lan_tre_90_ngay"] > 0) |
    (df["ty_le_su_dung_tin_dung"] > 0.8)
]

# Cách 2 — De Morgan (bạn tự viết)
canh_bao_2 = df[~((df['so_lan_tre_90_ngay'] == 0) & (df['ty_le_su_dung_tin_dung'] <= 0.8))]

print(f"Cách 1: {len(canh_bao_1):,}")
print(f"Cách 2: {len(canh_bao_2):,}")
print(f"Verify: {len(canh_bao_1) == len(canh_bao_2)}")

Cách 1: 28,373
Cách 2: 28,373
Verify: True


In [12]:
# Lấy nhóm trẻ + thêm cột nhãn rủi ro
# Bạn tự hoàn thành — nhớ .copy() và .loc
df_young = df[(df["tuoi"] >= 20) & (df["tuoi"] <= 40)].copy()
df_young.loc[:, "nhom_rui_ro"] = "Trẻ — cần theo dõi"

# Verify: in 3 hàng đầu, chỉ 3 cột: tuoi, vo_no, nhom_rui_ro
print(df_young[['tuoi','vo_no','nhom_rui_ro']].head(3))

   tuoi  vo_no         nhom_rui_ro
2    40      0  Trẻ — cần theo dõi
3    38      0  Trẻ — cần theo dõi
4    30      0  Trẻ — cần theo dõi
